# Attention Reallocation MMRAG

This notebook follows `Base_MMRAG.ipynb`, but adds an adaptive attention reallocation stage to reduce context bias. The flow is:

1. Retrieve image and text evidence separately with CLIP.
2. Estimate modality confidence from the top-k retrieval scores.
3. Reallocate attention by boosting the image score only when the visual evidence is strong or competitive.
4. Pass the visual weight into generation through `visual_token_boost` so the model relies more on the image when appropriate.

In [ ]:
# Uncomment this in a fresh notebook runtime.
%pip install -q torch torchvision faiss-cpu transformers pillow pandas numpy sentence-transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.1 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import json
import sys

import faiss
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data.json").exists() else NOTEBOOK_DIR.parent
MMRAG_DIR = PROJECT_ROOT / "MMRAG"
DATA_PATH = PROJECT_ROOT / "data.json"
IMAGE_DIR = PROJECT_ROOT / "images"
OUTPUT_PATH = MMRAG_DIR / "attn_realloc_mmrag_results.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

with DATA_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

clip_device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(clip_device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f"Loaded {len(data)} samples from {DATA_PATH}")
print(f"CLIP device: {clip_device}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loaded 79 samples from /content/data.json
CLIP device: cpu


In [ ]:
import numpy as np
from PIL import Image

image_embeddings = []
text_embeddings = []

for item in data:
    # ---- Image embedding ----
    try:
        image = Image.open("/content/images/" + item["image"]).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt").to(clip_device)

        with torch.no_grad():
            # Explicitly get the vision model output and extract pooler_output tensor
            outputs = clip_model.get_image_features(**inputs)
            # If get_image_features returns the tensor directly, use it; otherwise check for pooler_output
            img_emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

        image_embeddings.append(img_emb.cpu().numpy().flatten())

        # ---- Text embedding ----
        text = item["question"] + " " + item["context"]
        inputs = clip_processor(text=[text], return_tensors="pt", padding=True).to(clip_device)

        with torch.no_grad():
            # Explicitly get the text model output and extract pooler_output tensor
            outputs = clip_model.get_text_features(**inputs)
            txt_emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

        text_embeddings.append(txt_emb.cpu().numpy().flatten())
    except Exception as e:
        print(f"Error processing item {item.get('id')}: {e}")

image_embeddings = np.array(image_embeddings).astype("float32")
text_embeddings = np.array(text_embeddings).astype("float32")

print(image_embeddings)
print(text_embeddings)

[[-0.01368057  0.24527277  0.30411446 ...  1.4390602   0.48839647
   0.2716202 ]
 [-0.01368057  0.24527277  0.30411446 ...  1.4390602   0.48839647
   0.2716202 ]
 [-0.01368057  0.24527277  0.30411446 ...  1.4390602   0.48839647
   0.2716202 ]
 ...
 [ 0.08258414  0.23911664 -0.10811072 ...  1.3782201  -0.19505122
   0.00277483]
 [ 0.08258414  0.23911664 -0.10811072 ...  1.3782201  -0.19505122
   0.00277483]
 [ 0.08258414  0.23911664 -0.10811072 ...  1.3782201  -0.19505122
   0.00277483]]
[[-0.1578277   0.14341734 -0.2622878  ... -0.83291215  0.09503196
   0.41669247]
 [ 0.02373292  0.08180702 -0.08711476 ... -0.84396005 -0.05094998
   0.17452495]
 [ 0.11001205  0.12505916  0.15019448 ... -0.71186453 -0.28102034
   0.18373841]
 ...
 [ 0.26128715  0.3843072   0.1654998  ...  0.7640654   0.07012813
   0.08032955]
 [-0.04629003  0.23034735  0.08900556 ...  0.7548707   0.10126726
   0.34830835]
 [ 0.2389374   0.38032022 -0.04228793 ...  0.62998223  0.41798183
   0.1401931 ]]


In [ ]:
dim = image_embeddings.shape[1]

image_index = faiss.IndexFlatIP(dim)
text_index = faiss.IndexFlatIP(dim)

image_index.add(image_embeddings)
text_index.add(text_embeddings)


def retrieve_with_scores(query, top_k=3):
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(clip_device)

    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
        # Extract the actual tensor, handling BaseModelOutputWithPooling if present
        query_emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

    query_emb = query_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_emb)

    D_img, I_img = image_index.search(query_emb, top_k)
    D_txt, I_txt = text_index.search(query_emb, top_k)
    return D_img, I_img, D_txt, I_txt

## Adaptive Attention Reallocation

Instead of choosing the winning modality from only the single best score, this notebook summarizes the top-k evidence for each modality, estimates confidence, and then boosts the image side only when the image evidence is close enough to compete. That gives the image branch a fairer chance without blindly overriding strong text evidence.

In [ ]:
def normalize_similarity(scores):
    scores = np.asarray(scores, dtype="float32")
    return np.clip((scores + 1.0) / 2.0, 0.0, 1.0)


def summarize_modality(scores):
    normalized_scores = normalize_similarity(scores)
    rank_weights = np.linspace(1.0, 0.4, num=len(normalized_scores), dtype="float32")
    weighted_mean = float(np.average(normalized_scores, weights=rank_weights))
    margin = float(max(normalized_scores[0] - normalized_scores[1], 0.0)) if len(normalized_scores) > 1 else float(normalized_scores[0])
    confidence = (0.7 * weighted_mean) + (0.3 * margin)
    return {
        "normalized_scores": normalized_scores,
        "top1": float(normalized_scores[0]),
        "margin": margin,
        "confidence": confidence,
    }


def reallocate_attention(
    D_img,
    D_txt,
    image_prior=1.5,
    temperature=0.20,
    closeness_tau=0.15,
    boost_strength=2.2,
):
    image_stats = summarize_modality(D_img[0])
    text_stats = summarize_modality(D_txt[0])

    logits = np.array(
        [
            image_stats["confidence"] * image_prior,
            text_stats["confidence"],
        ],
        dtype="float32",
    )
    logits = logits / temperature
    logits = logits - logits.max()
    weights = np.exp(logits)
    lambda_image, lambda_text = (weights / weights.sum()).tolist()

    gap = max(text_stats["top1"] - image_stats["top1"], 0.0)
    closeness = float(np.exp(-gap / closeness_tau))

    image_boost = 1.0 + (boost_strength * lambda_image * closeness)
    text_scale = 0.80 + (0.15 * lambda_text)

    reallocated_image_score = image_stats["top1"] * image_boost
    reallocated_text_score = text_stats["top1"] * text_scale

    return {
        "lambda_image": float(lambda_image),
        "lambda_text": float(lambda_text),
        "image_boost": float(image_boost),
        "closeness": closeness,
        "image_stats": image_stats,
        "text_stats": text_stats,
        "reallocated_image_score": float(reallocated_image_score),
        "reallocated_text_score": float(reallocated_text_score),
    }


def select_best_modality(
    query,
    top_k=3,
    image_prior=1.30,
    temperature=0.20,
    closeness_tau=0.10,
    boost_strength=1.75,
):
    D_img, I_img, D_txt, I_txt = retrieve_with_scores(query, top_k=top_k)
    scores = reallocate_attention(
        D_img,
        D_txt,
        image_prior=image_prior,
        temperature=temperature,
        closeness_tau=closeness_tau,
        boost_strength=boost_strength,
    )

    if scores["reallocated_image_score"] >= scores["reallocated_text_score"]:
        best_idx = int(I_img[0][0])
        chosen_type = "image"
    else:
        best_idx = int(I_txt[0][0])
        chosen_type = "text"

    return {
        "type": chosen_type,
        "item": data[best_idx],
        "selected_index": best_idx,
        "scores": scores,
        "raw": {
            "D_img": D_img,
            "I_img": I_img,
            "D_txt": D_txt,
            "I_txt": I_txt,
        },
    }

In [ ]:
query = "On which day were the push-ups the lowest?"
selected = select_best_modality(query, top_k=3)
selected

{'type': 'image',
 'item': {'id': 9,
  'image': 'test3.jpg',
  'question': 'How many push-ups were done on Saturday?',
  'context': 'The weekend shows a decline with fewer than 10 push-ups. As the week winds down, the physical output drops significantly. This suggests a shift toward rest and recovery during the non-working days.',
  'answer': '20'},
 'selected_index': 8,
 'scores': {'lambda_image': 0.740774929523468,
  'lambda_text': 0.259225070476532,
  'image_boost': 2.296356126666069,
  'closeness': 1.0,
  'image_stats': {'normalized_scores': array([1., 1., 1.], dtype=float32),
   'top1': 1.0,
   'margin': 0.0,
   'confidence': 0.7},
  'text_stats': {'normalized_scores': array([1., 1., 1.], dtype=float32),
   'top1': 1.0,
   'margin': 0.0,
   'confidence': 0.7},
  'reallocated_image_score': 2.296356126666069,
  'reallocated_text_score': 0.8388837605714798},
 'raw': {'D_img': array([[3.136881, 3.136881, 3.136881]], dtype=float32),
  'I_img': array([[8, 7, 6]]),
  'D_txt': array([[9.3

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct", dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

def image_response(query, image):
  messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image,
              },
              {"type": "text", "text": query},
          ],
      }
  ]

  # Preparation for inference
  inputs = processor.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_dict=True,
      return_tensors="pt"
  )
  inputs = inputs.to(clip_device)

  # Inference: Generation of the output
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_ids_trimmed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  return output_text

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
results = []

for item in data:
    question = item["question"]
    actual_answer = item["answer"]
    example_id = item["id"]

    selected = select_best_modality(question, top_k=3)
    score_bundle = selected["scores"]

    if selected["type"] == "image":
        image_path = IMAGE_DIR / selected["item"]["image"]
        image = Image.open(image_path).convert("RGB")
        mmrag_answer = image_response(question, image)[0]
    else:
        mmrag_answer = selected["item"]["context"]

    results.append(
        {
            "id": example_id,
            "question": question,
            "actual_ans": actual_answer,
            "selected_modality": selected["type"],
            "retrieved_id": selected["item"]["id"],
            "lambda_image": score_bundle["lambda_image"],
            "lambda_text": score_bundle["lambda_text"],
            "image_boost": score_bundle["image_boost"],
            "reallocated_image_score": score_bundle["reallocated_image_score"],
            "reallocated_text_score": score_bundle["reallocated_text_score"],
            "mmrag_ans": mmrag_answer,
        }
    )
    print(f"Query Processed : {example_id} || Lambda image : {score_bundle["lambda_image"]} || lambda text : {score_bundle["lambda_text"]}")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_PATH}")
df.head()